<a href="https://colab.research.google.com/github/MaynineHuang/COMPSCI764_RAG/blob/main/notebooks/Retrieval_RAGBaseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [73]:
!git clone https://github.com/MaynineHuang/COMPSCI764_RAG.git
%cd COMPSCI764_RAG

Cloning into 'COMPSCI764_RAG'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 86 (delta 39), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (86/86), 78.61 KiB | 1.60 MiB/s, done.
Resolving deltas: 100% (39/39), done.
/content/COMPSCI764_RAG/COMPSCI764_RAG/COMPSCI764_RAG/COMPSCI764_RAG


In [74]:
!pip install -q datasets sentence-transformers

In [75]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util

In [76]:
#load the data
hotpot = load_dataset(
        "hotpotqa/hotpot_qa",
        "distractor",
        split="validation[:5]"
         )
print(hotpot)

Dataset({
    features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
    num_rows: 5
})


In [77]:
#load the embedding model
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
retrieval_model = SentenceTransformer(MODEL_NAME)
print("Model loaded:", MODEL_NAME)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded: sentence-transformers/all-MiniLM-L6-v2


In [78]:
example = hotpot[0]
print("Question:")
print(example["question"])
print("\nGold answer:")
print(example["answer"])
print("\nGold supporting facts:")
print(example["supporting_facts"])

Question:
Were Scott Derrickson and Ed Wood of the same nationality?

Gold answer:
yes

Gold supporting facts:
{'title': ['Scott Derrickson', 'Ed Wood'], 'sent_id': [0, 0]}


In [79]:
#从这一步, get the question
titles = example["context"]["title"]
sentence_lists = example["context"]["sentences"]
passages = []
for title, sentences in zip(titles, sentence_lists):
     passage = title + ". " + " ".join(sentences)
     passages.append(passage)
print("Number of passages:", len(passages))

Number of passages: 10


In [80]:
#get the passages
question_embedding = retrieval_model.encode(
    example["question"],
    convert_to_tensor=True
)
passage_embeddings = retrieval_model.encode(
    passages,
    convert_to_tensor=True
)
print("Question embedding shape:", question_embedding.shape)
print("Passage embeddings shape:", passage_embeddings.shape)

Question embedding shape: torch.Size([384])
Passage embeddings shape: torch.Size([10, 384])


In [81]:
#Calculate cosine similarity
scores = util.cos_sim(
    question_embedding,
    passage_embeddings
)[0]
print(scores)

tensor([0.4753, 0.2636, 0.2922, 0.1315, 0.5520, 0.0636, 0.3287, 0.0818, 0.3027,
        0.0329])


In [82]:
#rank all passages
ranked_indices = scores.argsort(descending=True)
for rank, idx in enumerate(ranked_indices):
   idx = int(idx)
   print("\nRank:", rank + 1)
   print("Score:", round(float(scores[idx]), 4))
   print("Title:", titles[idx])


Rank: 1
Score: 0.552
Title: Ed Wood

Rank: 2
Score: 0.4753
Title: Ed Wood (film)

Rank: 3
Score: 0.3287
Title: Adam Collis

Rank: 4
Score: 0.3027
Title: Conrad Brooks

Rank: 5
Score: 0.2922
Title: Woodson, Arkansas

Rank: 6
Score: 0.2636
Title: Scott Derrickson

Rank: 7
Score: 0.1315
Title: Tyler Bates

Rank: 8
Score: 0.0818
Title: Sinister (film)

Rank: 9
Score: 0.0636
Title: Deliver Us from Evil (2014 film)

Rank: 10
Score: 0.0329
Title: Doctor Strange (2016 film)


In [83]:
#keep only the top 8 passages
TOP_K = 8
top_indices = ranked_indices[:TOP_K]
retrieved_titles = []
print("TOP 5 RETRIEVED PASSAGES")
for rank, idx in enumerate(top_indices):
  idx = int(idx)

  retrieved_titles.append(titles[idx])

  print("\nRank:", rank + 1)
  print("Score:", round(float(scores[idx]), 4))
  print("Title:", titles[idx])

TOP 5 RETRIEVED PASSAGES

Rank: 1
Score: 0.552
Title: Ed Wood

Rank: 2
Score: 0.4753
Title: Ed Wood (film)

Rank: 3
Score: 0.3287
Title: Adam Collis

Rank: 4
Score: 0.3027
Title: Conrad Brooks

Rank: 5
Score: 0.2922
Title: Woodson, Arkansas

Rank: 6
Score: 0.2636
Title: Scott Derrickson

Rank: 7
Score: 0.1315
Title: Tyler Bates

Rank: 8
Score: 0.0818
Title: Sinister (film)


In [84]:
#measure supporting-passage evidene recall
gold_support_titles = set(example["supporting_facts"]["title"])
retrieved_title_set = set(retrieved_titles)

found_titles = gold_support_titles.intersection(retrieved_title_set)

evidence_recall = len(found_titles) / len(gold_support_titles)

print("Gold supporting titles:", gold_support_titles)
print("Retrieved titles:", retrieved_title_set)
print("Found supporting titles:", found_titles)
print("Evidence recall:", evidence_recall)

Gold supporting titles: {'Ed Wood', 'Scott Derrickson'}
Retrieved titles: {'Adam Collis', 'Conrad Brooks', 'Tyler Bates', 'Scott Derrickson', 'Ed Wood', 'Woodson, Arkansas', 'Sinister (film)', 'Ed Wood (film)'}
Found supporting titles: {'Ed Wood', 'Scott Derrickson'}
Evidence recall: 1.0


In [85]:
#define the retrieve_top_k
def retrieve_top_k(example, model, top_k=8):
  question = example["question"]
  titles = example["context"]["title"]
  sentence_lists = example["context"]["sentences"]

  passages = [
      title + ". " + " ".join(sentences)
      for title, sentences in zip(titles, sentence_lists)
  ]
  question_embedding = model.encode(
      question,
      convert_to_tensor=True
  )

  passage_embeddings = model.encode(
      passages,
      convert_to_tensor=True
  )

  scores = util.cos_sim(
      question_embedding,
      passage_embeddings    )[0]

  ranked_indices = scores.argsort(descending=True)
  top_indices = ranked_indices[:top_k]

  results = []

  for rank, idx in enumerate(top_indices):
    idx = int(idx)
    results.append({
        "rank": rank + 1,
        "title": titles[idx],
        "passage": passages[idx],
        "score": float(scores[idx])
        })

  return results

In [86]:
#test the function on example1
results = retrieve_top_k(
    hotpot[0],
    retrieval_model,
    top_k=8
)
for result in results:
  print(
      result["rank"],
      round(result["score"], 4),
      result["title"]
  )

1 0.552 Ed Wood
2 0.4753 Ed Wood (film)
3 0.3287 Adam Collis
4 0.3027 Conrad Brooks
5 0.2922 Woodson, Arkansas
6 0.2636 Scott Derrickson
7 0.1315 Tyler Bates
8 0.0818 Sinister (film)


In [87]:
for i in range(len(hotpot)):
  example = hotpot[i]

  results = retrieve_top_k(
      example,
      retrieval_model,
      top_k=8
  )

  retrieved_titles = {
      result["title"]
      for result in results
  }

  gold_titles = set(
      example["supporting_facts"]["title"]
  )

  found = gold_titles.intersection(retrieved_titles)

  recall = len(found) / len(gold_titles)

  print("\n" + "=" * 70)
  print("EXAMPLE", i + 1)
  print("Question:", example["question"])
  print("Gold answer:", example["answer"])
  print("Gold supporting titles:", gold_titles)

  print("\nTop 8:")
  for result in results:
    print(
        result["rank"],
        "|",
        round(result["score"], 4),
        "|",
        result["title"]
    )

  print("\nEvidence recall:", recall)


EXAMPLE 1
Question: Were Scott Derrickson and Ed Wood of the same nationality?
Gold answer: yes
Gold supporting titles: {'Ed Wood', 'Scott Derrickson'}

Top 8:
1 | 0.552 | Ed Wood
2 | 0.4753 | Ed Wood (film)
3 | 0.3287 | Adam Collis
4 | 0.3027 | Conrad Brooks
5 | 0.2922 | Woodson, Arkansas
6 | 0.2636 | Scott Derrickson
7 | 0.1315 | Tyler Bates
8 | 0.0818 | Sinister (film)

Evidence recall: 1.0

EXAMPLE 2
Question: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
Gold answer: Chief of Protocol
Gold supporting titles: {'Shirley Temple', 'Kiss and Tell (1945 film)'}

Top 8:
1 | 0.6683 | Kiss and Tell (1945 film)
2 | 0.5741 | A Kiss for Corliss
3 | 0.5452 | Meet Corliss Archer (TV series)
4 | 0.3984 | Janet Waldo
5 | 0.3342 | Meet Corliss Archer
6 | 0.305 | Shirley Temple
7 | 0.2271 | Charles Craft
8 | 0.1181 | Secretary of State for Constitutional Affairs

Evidence recall: 1.0

EXAMPLE 3
Question: What science fantasy young adult seri

In [88]:
# Final Retrieval V1 validation on 20 examples

hotpot_20 = load_dataset(
    "hotpotqa/hotpot_qa",
    "distractor",
    split="validation[:20]"
)

recalls = []

for i in range(20):
    example = hotpot_20[i]

    results = retrieve_top_k(
        example,
        retrieval_model,
        top_k=8
    )

    retrieved_titles = {
        result["title"]
        for result in results
    }

    gold_titles = set(
        example["supporting_facts"]["title"]
    )

    found_titles = gold_titles.intersection(retrieved_titles)

    recall = len(found_titles) / len(gold_titles)
    recalls.append(recall)

    print(
        f"Example {i + 1}: "
        f"Recall@5 = {recall:.2f} | "
        f"Found {len(found_titles)}/{len(gold_titles)} supporting passages"
    )

average_recall = sum(recalls) / len(recalls)

full_recall_count = sum(
    recall == 1.0
    for recall in recalls
)

print("\n" + "=" * 60)
print("RETRIEVAL V1 - 20 EXAMPLE SUMMARY")
print("=" * 60)
print("Average Evidence Recall@5:", round(average_recall, 3))
print("Questions with full supporting evidence:", full_recall_count, "/ 20")
print(
    "Full evidence retrieval rate:",
    round(full_recall_count / 20, 3)
)

Example 1: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 2: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 3: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 4: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 5: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 6: Recall@5 = 0.50 | Found 1/2 supporting passages
Example 7: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 8: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 9: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 10: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 11: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 12: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 13: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 14: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 15: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 16: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 17: Recall@5 = 0.50 | Found 1/2 supportin

In [89]:
# Retrieval V1 - Final validation on 20 HotpotQA examples

!pip install -q datasets sentence-transformers

from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util


# Settings
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K = 8
NUM_EXAMPLES = 20


# Load 20 validation examples
hotpot_20 = load_dataset(
    "hotpotqa/hotpot_qa",
    "distractor",
    split=f"validation[:{NUM_EXAMPLES}]"
)

print("Loaded examples:", len(hotpot_20))


# Load retrieval model
retrieval_model = SentenceTransformer(MODEL_NAME)

print("Model loaded:", MODEL_NAME)


# Retrieval function
def retrieve_top_k(example, model, top_k=5):
    question = example["question"]
    titles = example["context"]["title"]
    sentence_lists = example["context"]["sentences"]

    passages = [
        title + ". " + " ".join(sentences)
        for title, sentences in zip(titles, sentence_lists)
    ]

    question_embedding = model.encode(
        question,
        convert_to_tensor=True,
        show_progress_bar=False
    )

    passage_embeddings = model.encode(
        passages,
        convert_to_tensor=True,
        show_progress_bar=False
    )

    scores = util.cos_sim(
        question_embedding,
        passage_embeddings
    )[0]

    top_k = min(top_k, len(passages))

    ranked_indices = scores.argsort(descending=True)
    top_indices = ranked_indices[:top_k]

    results = []

    for rank, idx in enumerate(top_indices):
        idx = int(idx)

        results.append({
            "rank": rank + 1,
            "title": titles[idx],
            "passage": passages[idx],
            "score": float(scores[idx])
        })

    return results


# Evaluate Retrieval V1
recalls = []
full_recall_count = 0

for i in range(NUM_EXAMPLES):
    example = hotpot_20[i]

    results = retrieve_top_k(
        example,
        retrieval_model,
        top_k=TOP_K
    )

    retrieved_titles = {
        result["title"]
        for result in results
    }

    gold_titles = set(
        example["supporting_facts"]["title"]
    )

    found_titles = gold_titles.intersection(retrieved_titles)

    recall = len(found_titles) / len(gold_titles)
    recalls.append(recall)

    if recall == 1.0:
        full_recall_count += 1

    print(
        f"Example {i + 1:02d} | "
        f"Recall@{TOP_K}: {recall:.2f} | "
        f"Found: {len(found_titles)}/{len(gold_titles)}"
    )


# Final summary
average_recall = sum(recalls) / len(recalls)
full_retrieval_rate = full_recall_count / NUM_EXAMPLES

print("\n" + "=" * 60)
print("RETRIEVAL V1 - 20 EXAMPLE SUMMARY")
print("=" * 60)

print(
    f"Average Evidence Recall@{TOP_K}: "
    f"{average_recall:.3f}"
)

print(
    "Questions with full supporting evidence:",
    f"{full_recall_count}/{NUM_EXAMPLES}"
)

print(
    "Full evidence retrieval rate:",
    f"{full_retrieval_rate:.3f}"
)

Loaded examples: 20


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded: sentence-transformers/all-MiniLM-L6-v2
Example 01 | Recall@8: 1.00 | Found: 2/2
Example 02 | Recall@8: 1.00 | Found: 2/2
Example 03 | Recall@8: 1.00 | Found: 2/2
Example 04 | Recall@8: 1.00 | Found: 2/2
Example 05 | Recall@8: 1.00 | Found: 2/2
Example 06 | Recall@8: 0.50 | Found: 1/2
Example 07 | Recall@8: 1.00 | Found: 2/2
Example 08 | Recall@8: 1.00 | Found: 2/2
Example 09 | Recall@8: 1.00 | Found: 2/2
Example 10 | Recall@8: 1.00 | Found: 2/2
Example 11 | Recall@8: 1.00 | Found: 2/2
Example 12 | Recall@8: 1.00 | Found: 2/2
Example 13 | Recall@8: 1.00 | Found: 2/2
Example 14 | Recall@8: 1.00 | Found: 2/2
Example 15 | Recall@8: 1.00 | Found: 2/2
Example 16 | Recall@8: 1.00 | Found: 2/2
Example 17 | Recall@8: 0.50 | Found: 1/2
Example 18 | Recall@8: 1.00 | Found: 2/2
Example 19 | Recall@8: 0.50 | Found: 1/2
Example 20 | Recall@8: 1.00 | Found: 2/2

RETRIEVAL V1 - 20 EXAMPLE SUMMARY
Average Evidence Recall@8: 0.925
Questions with full supporting evidence: 17/20
Full evidenc

In [90]:
# Compare different Top-k values before freezing retrieval

TOP_K_VALUES = [5, 6, 7, 8]

for top_k in TOP_K_VALUES:
    recalls = []
    full_recall_count = 0

    for i in range(len(hotpot_20)):
        example = hotpot_20[i]

        results = retrieve_top_k(
            example,
            retrieval_model,
            top_k=top_k
        )

        retrieved_titles = {
            result["title"]
            for result in results
        }

        gold_titles = set(
            example["supporting_facts"]["title"]
        )

        found_titles = gold_titles.intersection(retrieved_titles)

        recall = len(found_titles) / len(gold_titles)
        recalls.append(recall)

        if recall == 1.0:
            full_recall_count += 1

    average_recall = sum(recalls) / len(recalls)
    full_retrieval_rate = full_recall_count / len(hotpot_20)

    print(
        f"Top-{top_k} | "
        f"Average Recall: {average_recall:.3f} | "
        f"Full Evidence: {full_recall_count}/20 | "
        f"Full Evidence Rate: {full_retrieval_rate:.3f}"
    )

Top-5 | Average Recall: 0.775 | Full Evidence: 11/20 | Full Evidence Rate: 0.550
Top-6 | Average Recall: 0.825 | Full Evidence: 13/20 | Full Evidence Rate: 0.650
Top-7 | Average Recall: 0.875 | Full Evidence: 15/20 | Full Evidence Rate: 0.750
Top-8 | Average Recall: 0.925 | Full Evidence: 17/20 | Full Evidence Rate: 0.850


In [91]:
# Oct 15 2026 new-update meihan
!pip install -q transformers sentencepiece accelerate


In [92]:

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

GENERATOR_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(GENERATOR_NAME)

generator_model = AutoModelForSeq2SeqLM.from_pretrained(
    GENERATOR_NAME
)

device = "cuda" if torch.cuda.is_available() else "cpu"
generator_model = generator_model.to(device)
generator_model.eval()

print("Generator loaded:", GENERATOR_NAME)
print("Device:", device)


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Generator loaded: google/flan-t5-base
Device: cpu


In [93]:

TOP_K = 8

def generate_rag_answer(question, retrieved_results):
    """
    Generate an answer using the question and retrieved passages.
    No attack injection and no defence are applied.
    """

    context = "\n\n".join(
        [
            f"Passage {item['rank']} "
            f"(Title: {item['title']}):\n{item['passage']}"
            for item in retrieved_results
        ]
    )

    prompt = f"""Answer the question using the context below.
Give a short and direct answer.
Do not add explanations unless necessary.
If the answer cannot be found in the context, say you do not know.

Context:
{context}

Question: {question}

Answer:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(device)

    with torch.no_grad():
        outputs = generator_model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False
        )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer.strip()


In [94]:

# Select one example from the validation dataset
example = hotpot_20[0]

# Retrieve the top 8 passages
retrieved_results = retrieve_top_k(
    example,
    retrieval_model,
    top_k=TOP_K
)

# Generate the answer
predicted_answer = generate_rag_answer(
    example["question"],
    retrieved_results
)

print("Question:")
print(example["question"])

print("\nGold answer:")
print(example["answer"])

print("\nPredicted answer:")
print(predicted_answer)

print("\nRetrieved passages:")
for item in retrieved_results:
    print(
        item["rank"],
        "|",
        round(item["score"], 4),
        "|",
        item["title"]
    )


Question:
Were Scott Derrickson and Ed Wood of the same nationality?

Gold answer:
yes

Predicted answer:
Arkansas town.

Retrieved passages:
1 | 0.552 | Ed Wood
2 | 0.4753 | Ed Wood (film)
3 | 0.3287 | Adam Collis
4 | 0.3027 | Conrad Brooks
5 | 0.2922 | Woodson, Arkansas
6 | 0.2636 | Scott Derrickson
7 | 0.1315 | Tyler Bates
8 | 0.0818 | Sinister (film)


In [95]:

import re
import string
from collections import Counter

def normalize_answer(text):
    """Normalize an answer for HotpotQA-style evaluation."""
    text = text.lower()
    text = "".join(
        char for char in text
        if char not in string.punctuation
    )
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = " ".join(text.split())
    return text


def exact_match_score(prediction, reference):
    return int(
        normalize_answer(prediction)
        == normalize_answer(reference)
    )


def token_f1_score(prediction, reference):
    pred_tokens = normalize_answer(prediction).split()
    ref_tokens = normalize_answer(reference).split()

    if not pred_tokens or not ref_tokens:
        return float(pred_tokens == ref_tokens)

    common = Counter(pred_tokens) & Counter(ref_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(ref_tokens)

    return 2 * precision * recall / (precision + recall)


In [96]:

baseline_results = []

for i, example in enumerate(hotpot_20):
    question = example["question"]
    gold_answer = example["answer"]

    # Step 1: Retrieve top-8 passages
    retrieved = retrieve_top_k(
        example,
        retrieval_model,
        top_k=TOP_K
    )

    # Step 2: Generate an answer without attack or defence
    prediction = generate_rag_answer(
        question,
        retrieved
    )

    # Step 3: Evaluate answer quality
    em = exact_match_score(prediction, gold_answer)
    f1 = token_f1_score(prediction, gold_answer)

    # Step 4: Evaluate supporting-passage recall
    retrieved_titles = {
        item["title"] for item in retrieved
    }
    gold_titles = set(example["supporting_facts"]["title"])

    if gold_titles:
        evidence_recall = (
            len(gold_titles & retrieved_titles)
            / len(gold_titles)
        )
    else:
        evidence_recall = 0.0

    baseline_results.append({
        "id": example["id"],
        "question": question,
        "gold_answer": gold_answer,
        "prediction": prediction,
        "EM": em,
        "Token_F1": f1,
        "Evidence_Recall": evidence_recall
    })

    print(
        f"Example {i + 1}/{len(hotpot_20)} | "
        f"EM={em} | "
        f"F1={f1:.3f} | "
        f"Evidence Recall={evidence_recall:.3f}"
    )

print("\n===== CLEAN RAG BASELINE =====")

print(
    "Average EM:",
    round(
        sum(r["EM"] for r in baseline_results)
        / len(baseline_results),
        3
    )
)

print(
    "Average Token F1:",
    round(
        sum(r["Token_F1"] for r in baseline_results)
        / len(baseline_results),
        3
    )
)

print(
    "Average Evidence Recall:",
    round(
        sum(r["Evidence_Recall"] for r in baseline_results)
        / len(baseline_results),
        3
    )
)


Example 1/20 | EM=0 | F1=0.000 | Evidence Recall=1.000
Example 2/20 | EM=0 | F1=0.000 | Evidence Recall=1.000
Example 3/20 | EM=0 | F1=0.143 | Evidence Recall=1.000
Example 4/20 | EM=0 | F1=0.000 | Evidence Recall=1.000
Example 5/20 | EM=0 | F1=0.000 | Evidence Recall=1.000
Example 6/20 | EM=0 | F1=0.000 | Evidence Recall=0.500
Example 7/20 | EM=0 | F1=0.000 | Evidence Recall=1.000
Example 8/20 | EM=0 | F1=0.000 | Evidence Recall=1.000
Example 9/20 | EM=0 | F1=0.000 | Evidence Recall=1.000
Example 10/20 | EM=0 | F1=0.000 | Evidence Recall=1.000
Example 11/20 | EM=0 | F1=0.043 | Evidence Recall=1.000
Example 12/20 | EM=0 | F1=0.000 | Evidence Recall=1.000
Example 13/20 | EM=0 | F1=0.000 | Evidence Recall=1.000
Example 14/20 | EM=0 | F1=0.000 | Evidence Recall=1.000
Example 15/20 | EM=0 | F1=0.000 | Evidence Recall=1.000
Example 16/20 | EM=0 | F1=0.000 | Evidence Recall=1.000
Example 17/20 | EM=0 | F1=0.048 | Evidence Recall=0.500
Example 18/20 | EM=0 | F1=0.000 | Evidence Recall=1.000
E

In [97]:

import pandas as pd

# Save the existing baseline results
baseline_df = pd.DataFrame(baseline_results)

baseline_df.to_csv(
    "clean_rag_baseline.csv",
    index=False
)

print("Number of examples:", len(baseline_df))
print("\nAverage metrics:")
print(
    baseline_df[["EM", "Token_F1", "Evidence_Recall"]]
    .mean()
)

print("\nMissing values:")
print(
    baseline_df[
        ["id", "prediction", "EM", "Token_F1", "Evidence_Recall"]
    ].isna().sum()
)

print("\nSaved: clean_rag_baseline.csv")


Number of examples: 20

Average metrics:
EM                 0.000000
Token_F1           0.019015
Evidence_Recall    0.925000
dtype: float64

Missing values:
id                 0
prediction         0
EM                 0
Token_F1           0
Evidence_Recall    0
dtype: int64

Saved: clean_rag_baseline.csv


In [98]:

import pandas as pd

df = pd.DataFrame(baseline_results)

# Inspect questions that were not answered exactly correctly
errors = df[df["EM"] == 0][
    [
        "id",
        "question",
        "gold_answer",
        "prediction",
        "EM",
        "Token_F1",
        "Evidence_Recall"
    ]
]

print("Number of EM errors:", len(errors))
display(errors)


Number of EM errors: 20


,id,question,gold_answer,prediction,EM,Token_F1,Evidence_Recall
0,5a8b57f25542995d1e6f1371,Were Scott Derrickson and Ed Wood of the same ...,yes,Arkansas town.,0,0.000000,1.0
1,5a8c7595554299585d9e36b6,What government position was held by the woman...,Chief of Protocol,"The title character in ""Meet Corliss Archer"" i...",0,0.000000,1.0
2,5a85ea095542994775f606a8,"What science fantasy young adult series, told ...",Animorphs,The Hork-Bajir Chronicles is the second compan...,0,0.142857,1.0
3,5adbf0a255429947ff17385a,Are the Laleli Mosque and Esma Sultan Mansion ...,no,Sultan Ahmed Mosque is a historic mosque locat...,0,0.000000,1.0
4,5a8e3ea95542995a26add48d,"The director of the romantic comedy ""Big Stone...","Greenwich Village, New York City",Itsbynne Reel was no,0,0.000000,1.0
5,5abd94525542992ac4f382d2,2014 S/S is the debut album of a South Korean ...,YG Entertainment,Shinee World Award.,0,0.000000,0.5
6,5a85b2d95542997b5ce40028,Who was known by his stage name Aladin and hel...,Eenasul Fateh,""" Eldee is a diminutive for the Yoruba name ""O...",0,0.000000,1.0
7,5a87ab905542996e4f3088c1,The arena where the Lewiston Maineiacs played ...,"3,677 seated",The Boston University Terriers men's and women...,0,0.000000,1.0
8,5a7bbb64554299042af8f7cc,"Who is older, Annie Morton or Terry Richardson?",Terry Richardson,He has also shot portraits for Yves Saint Laur...,0,0.000000,1.0
9,5a8db19d5542994ba4e3dd00,Are Local H and For Against both from the Unit...,yes,The album is the seventh studio album by Chica...,0,0.000000,1.0


In [99]:

# Check input token length for the first example
example = hotpot_20[0]

retrieved = retrieve_top_k(
    example,
    retrieval_model,
    top_k=8
)

context = "\n\n".join(
    [
        f"Passage {item['rank']} "
        f"(Title: {item['title']}):\n{item['passage']}"
        for item in retrieved
    ]
)

prompt = f"""Answer the question using the context below.
Give a short and direct answer.
Do not add explanations unless necessary.
If the answer cannot be found in the context, say you do not know.

Context:
{context}

Question: {example["question"]}

Answer:"""

encoded = tokenizer(
    prompt,
    truncation=False
)

print("Input token count:", len(encoded["input_ids"]))
print("Configured input limit:", 2048)
print("Exceeds limit:", len(encoded["input_ids"]) > 2048)


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1035 > 512). Running this sequence through the model will result in indexing errors


Input token count: 1035
Configured input limit: 2048
Exceeds limit: False
